# WBF重み調整
現在のベスト0.0982からさらに上を目指す

In [ ]:
import os
os.chdir(r'C:\compe')

import json
import numpy as np
import pandas as pd
import torch
import torchvision.transforms.functional as TF
from PIL import Image as PILImage
from ultralytics import YOLO, RTDETR
from rfdetr import RFDETRBase
from ensemble_boxes import weighted_boxes_fusion
from pathlib import Path
from tqdm import tqdm
from datetime import datetime

print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'なし')

with open(r'C:\compe\test_dataset.json') as f:
    test_data = json.load(f)

cat_ids = sorted([c['id'] for c in test_data['categories']])
yolo_to_category = cat_ids
fname_to_id    = {img['file_name']: img['id'] for img in test_data['images']}
img_id_to_info = {img['id']: img for img in test_data['images']}
TEST_DIR   = r'C:\compe\images\test'
test_files = sorted(os.listdir(TEST_DIR))
print(f'test画像数: {len(test_files)}')

In [ ]:
# モデル読み込み
model_yolom = YOLO(r'C:\compe\runs\pseudo2\weights\best.pt')
print('yolo26m loaded')

model_yolol = YOLO(r'C:\Users\tamkn\Downloads\best.pt')
print('yolo26l loaded')

model_rtdetr = RTDETR(r'C:\compe\runs\rtdetr_pseudo\weights\best.pt')
print('RT-DETR loaded')

model_yolox = YOLO(r'C:\compe\runs\yolox_pseudo\weights\best.pt')
print('yolo26x loaded')

model_rfdetr = RFDETRBase(
    pretrain_weights=r'C:\compe\runs\rfdetr4_pseudo\checkpoint_best_total.pth',
    num_classes=32
)
model_rfdetr.optimize_for_inference()
print('RF-DETR loaded')

In [ ]:
# 推論関数
CONF             = 0.001
IOU              = 0.6
MAX_DET          = 1000
WBF_IOU          = 0.55
WBF_SKIP_BOX_THR = 0.005
RFDETR_THR       = 0.1

TTA_PATTERNS = [
    (False, 1024),
    (True,  1024),
    (False,  896),
]

def predict_yolo_tta(model, img_path, conf, iou):
    img_orig = PILImage.open(img_path).convert('RGB')
    all_boxes, all_scores, all_labels = [], [], []
    for do_flip, sz in TTA_PATTERNS:
        img = img_orig.copy()
        if do_flip:
            img = TF.hflip(img)
        result = model.predict(
            source=img, conf=conf, iou=iou,
            max_det=MAX_DET, imgsz=sz,
            verbose=False, half=True,
        )[0]
        if len(result.boxes) == 0:
            continue
        boxes  = result.boxes.xyxyn.cpu().numpy().tolist()
        scores = result.boxes.conf.cpu().numpy().tolist()
        labels = result.boxes.cls.cpu().numpy().astype(int).tolist()
        if do_flip:
            boxes = [[1-x2, y1, 1-x1, y2] for x1, y1, x2, y2 in boxes]
        boxes = [[min(max(v, 0.0), 1.0) for v in box] for box in boxes]
        all_boxes.append(boxes)
        all_scores.append(scores)
        all_labels.append(labels)
    if not all_boxes:
        return [], [], []
    boxes_f, scores_f, labels_f = weighted_boxes_fusion(
        all_boxes, all_scores, all_labels,
        weights=[1.0] * len(all_boxes),
        iou_thr=WBF_IOU, skip_box_thr=WBF_SKIP_BOX_THR,
    )
    return boxes_f.tolist(), scores_f.tolist(), labels_f.tolist()

def predict_rfdetr(model, img_path, w, h, threshold):
    img = PILImage.open(img_path).convert('RGB')
    result = model.predict(img, threshold=threshold)
    if len(result) == 0:
        return [], [], []
    valid = result.class_id < 32
    if not valid.any():
        return [], [], []
    boxes  = result.xyxy[valid].astype(float)
    boxes[:, [0, 2]] /= w
    boxes[:, [1, 3]] /= h
    boxes  = np.clip(boxes, 0, 1).tolist()
    scores = result.confidence[valid].tolist()
    labels = result.class_id[valid].tolist()
    return boxes, scores, labels

print('推論関数定義完了')

In [ ]:
# 全モデルの予測を事前に取得（重み調整を高速化）
print('全モデルで推論中...')
all_preds = {}

for fname in tqdm(test_files):
    img_path = os.path.join(TEST_DIR, fname)
    image_id = fname_to_id.get(fname)
    if image_id is None:
        stem = Path(fname).stem
        for key in fname_to_id:
            if Path(key).stem == stem:
                image_id = fname_to_id[key]
                break
    if image_id is None:
        continue

    w = img_id_to_info[image_id]['width']
    h = img_id_to_info[image_id]['height']

    boxes_m,  scores_m,  labels_m  = predict_yolo_tta(model_yolom,  img_path, CONF, IOU)
    boxes_l,  scores_l,  labels_l  = predict_yolo_tta(model_yolol,  img_path, CONF, IOU)
    boxes_r,  scores_r,  labels_r  = predict_yolo_tta(model_rtdetr, img_path, CONF, IOU)
    boxes_rf, scores_rf, labels_rf = predict_rfdetr(model_rfdetr, img_path, w, h, RFDETR_THR)
    boxes_x,  scores_x,  labels_x  = predict_yolo_tta(model_yolox,  img_path, CONF, IOU)

    all_preds[fname] = {
        'image_id': image_id, 'w': w, 'h': h,
        'yolom':  (boxes_m,  scores_m,  labels_m),
        'yolol':  (boxes_l,  scores_l,  labels_l),
        'rtdetr': (boxes_r,  scores_r,  labels_r),
        'rfdetr': (boxes_rf, scores_rf, labels_rf),
        'yolox':  (boxes_x,  scores_x,  labels_x),
    }

print('推論完了！')

In [ ]:
# 重みを変えてsubmissionを作成する関数
def make_submission(weights, suffix):
    rows = []
    for fname, preds in all_preds.items():
        image_id = preds['image_id']
        w, h = preds['w'], preds['h']

        boxes_list, scores_list, labels_list, weights_used = [], [], [], []
        for key, wt in zip(['yolom', 'rtdetr', 'rfdetr', 'yolol', 'yolox'], weights):
            boxes, scores, labels = preds[key]
            if len(boxes) > 0:
                boxes_list.append(boxes)
                scores_list.append(scores)
                labels_list.append(labels)
                weights_used.append(wt)

        if not boxes_list:
            continue

        boxes_f, scores_f, labels_f = weighted_boxes_fusion(
            boxes_list, scores_list, labels_list,
            weights=weights_used,
            iou_thr=WBF_IOU, skip_box_thr=WBF_SKIP_BOX_THR,
        )

        for box, score, label in zip(boxes_f, scores_f, labels_f):
            x1, y1, x2, y2 = box
            rows.append({
                'image_id':    image_id,
                'category_id': yolo_to_category[int(label)],
                'bbox_x':      x1 * w,
                'bbox_y':      y1 * h,
                'bbox_width':  (x2 - x1) * w,
                'bbox_height': (y2 - y1) * h,
                'score':       float(score),
            })

    submission = pd.DataFrame(rows)
    submission['annotation_id'] = np.arange(len(submission))
    submission = submission[[
        'annotation_id', 'image_id', 'category_id',
        'bbox_x', 'bbox_y', 'bbox_width', 'bbox_height', 'score'
    ]]
    submission['score'] = submission['score'].clip(0, 1)
    path = rf'C:\compe\submission_{suffix}.csv'
    submission.to_csv(path, index=False)
    print(f'保存: {path} ({len(submission)}行)')
    return submission


# 試す重みのパターン（yolo26m : RT-DETR : RF-DETR : yolo26l）
# 試す重みのパターン（yolo26m : RT-DETR : RF-DETR : yolo26l : yolo26x）
weight_patterns = {
    'base':      [0.5, 0.5, 3.0, 2.0, 1.0],   # 現在のベスト
    'rtdetr3':   [0.5, 3.0, 3.0, 2.0, 1.0],   # RT-DETRを強く
    'rtdetr4':   [0.5, 4.0, 3.0, 2.0, 1.0],   # RT-DETRをさらに強く
    'rtdetr5':   [0.5, 5.0, 3.0, 2.0, 1.0],   # RT-DETRを最強に
    'no_yolom':  [0.0, 4.0, 3.0, 2.0, 1.0],   # yolo26m外す
    'yolox2':    [0.5, 3.0, 3.0, 2.0, 2.0],   # yolo26xを強く
    'rfdetr4':   [0.5, 3.0, 4.0, 2.0, 1.0],   # RF-DETRを強く
}
